# Modulo 01: Caracteristicas de la Vivienda y del Hogar (ENAHO 2025)

Este notebook ejecuta el pipeline automatizado y validado definido en `src/`:
- **Filtro geografico:** Lima y Callao (UBIGEO 07 y 15).
- **Filtro de cobertura:** Entrevistas completas y validas (RESULT in [1, 2]).
- **Resolucion de nulos estructurales:** Imputacion intra-vivienda para hogares secundarios (HOGAR > 11).
- **Ingenieria de Caracteristicas:** Consolidacion de tenencia y SUNARP (seguridad_tenencia).
- **Validaciones estrictas:** Esquema de columnas, recuento de filas y unicidad de llave primaria.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar directorio raiz del proyecto para importar src
ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name.startswith(('1031', '966')) else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.processors.modulo01 import Modulo01Processor

In [ ]:
# Instanciar el procesador y ejecutar el pipeline con validaciones automaticas
processor = Modulo01Processor(
    file_path='Enaho01-2025-100.csv',
    year=2025,
    output_path='Enaho01-2025-100_cleaned.csv'
)

df = processor.process()
display(df.shape)
display(df.head(10))

In [ ]:
# Reporte de Nulos y Calidad de Datos
null_summary = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().mean() * 100).round(2)
})
display(null_summary[null_summary['Nulos'] > 0] if (df.isnull().sum() > 0).any() else '¡0% NULOS EN TODAS LAS VARIABLES!')

In [ ]:
# Estadisticas Descriptivas
display(df.describe(include=[np.number]))
display(df.describe(include=['category']))

In [ ]:
# EDA Visual: Distribucion de Variables Categoricas
cat_cols = df.select_dtypes(include='category').columns.tolist()
n_cols = len(cat_cols)
n_plots_x = 3
n_plots_y = int(np.ceil(n_cols / n_plots_x))

fig, axes = plt.subplots(nrows=n_plots_y, ncols=n_plots_x, figsize=(16, 4 * n_plots_y))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df[col].value_counts().head(10).plot(kind='barh', ax=axes[i], color='steelblue', title=col)
    axes[i].set_xlabel('Cantidad de Hogares')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# EDA Visual: Variables Numericas
num_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(nrows=len(num_cols), ncols=2, figsize=(12, 3.5 * len(num_cols)))
if len(num_cols) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, col in enumerate(num_cols):
    df[col].plot(kind='hist', bins=15, ax=axes[i, 0], color='teal', title=f'Histograma: {col}')
    df[col].plot(kind='box', vert=False, ax=axes[i, 1], title=f'Boxplot: {col}')

plt.tight_layout()
plt.show()